# plotlywaterfall examples

Waterfall charts with [plotly](https://plotly.com/python/): grouping, stacked categories, totals and subtotals, colours, value labels.
Every example below is a single call to `Waterfall(...)` followed by `.get_fig()`, which returns a normal `plotly.graph_objects.Figure`.

Source: [github.com/docdru/plotlywaterfall](https://github.com/docdru/plotlywaterfall) · install with `pip install plotlywaterfall`

In [ ]:
import pandas as pd
import plotly.io as pio

from plotlywaterfall import Waterfall

pio.templates.default = "plotly_white"

## 1. A basic waterfall

Pass a DataFrame and the names of the x and y columns. Rows with the same `X` are summed. Without categories or groups, bars are coloured by direction (increase, decrease, total) and labelled with their change:

In [ ]:
df_cash = pd.DataFrame({
    "X": ["Product Sales", "Product Sales", "Services Sales", "Product R&D", "Product R&D", "Services COGS"],
    "Y": [3, 4, 1, -1, -2, -0.5],
    "category": ["Product A", "Product B", "Plugins", "Product A", "Product B", "Plugins"],
})
df_cash

In [ ]:
Waterfall(df_cash, x="X", y="Y", labels={"Y": "Cashflow [MUSD]"}).get_fig()

## 2. Totals and subtotals

`total=True` adds a total bar after the last x-value; `subtotals` adds intermediate totals after the given x-values:

In [ ]:
Waterfall(
    df_cash, x="X", y="Y", total=True, subtotals={"Services Sales": "Gross revenue"},
    labels={"Y": "Cashflow [MUSD]"},
).get_fig()

Subtotals can also be added afterwards with `add_subtotal`. The total can be negative:

In [ ]:
df_loss = df_cash.assign(Y=[3, 4, 1, -4, -5, -2])
wf = Waterfall(df_loss, x="X", y="Y", total=True, labels={"Y": "Cashflow [MUSD]"})
wf.add_subtotal("Services Sales", "Gross revenue")
wf.get_fig()

## 3. Stacked categories

With `category`, each bar is stacked from its categories, and totals keep the breakdown:

In [ ]:
Waterfall(df_cash, x="X", y="Y", category="category", total=True, labels={"Y": "Cashflow [MUSD]"}).get_fig()

## 4. Mixed signs within one bar

Categories of one bar can have different signs, e.g. a subsidy (+1) and a restructuring charge (−4) at the same `X`, or a total combining positive and negative categories. Such a bar is split into one narrow sub-step per category, walking from the level where the bar starts to where it ends. A light *net ghost* bar behind the sub-steps shows the net change:

In [ ]:
df_mixed = pd.DataFrame({
    "X": ["Revenue", "Costs", "Other", "Other"],
    "Y": [9, -7, 1, -4],
    "category": ["Operating", "Operating", "Operating", "One-off"],
})
Waterfall(df_mixed, x="X", y="Y", category="category", total=True).get_fig()

The order of the sub-steps follows the data; `step_order="positive_first"` or `"largest_first"` changes it. `net_ghost=False` hides the net bar (a dict of plotly marker properties styles it):

In [ ]:
Waterfall(df_mixed, x="X", y="Y", category="category", total=True, step_order="largest_first", net_ghost=False).get_fig()

`mixed_signs="error"` restores the strict behaviour of earlier versions:

In [ ]:
try:
    Waterfall(df_mixed, x="X", y="Y", category="category", mixed_signs="error")
except NotImplementedError as exc:
    print("NotImplementedError:", exc)

## 5. Groups

`group` draws one bar per group side by side, e.g. to compare plan and actual or several years. This DataFrame is used in the following sections:

In [ ]:
df = pd.DataFrame({
    "X": ["A", "B", "C"] * 4 + ["D", "D"],
    "Y": [4, 1, 8, 7, 3, 2] + [i - 1 for i in [4, 1, 8, 7, 3, 2]] + [8, 5],
    "category": ["one"] * 3 + ["two"] * 3 + ["one"] * 3 + ["two"] * 3 + ["two", "three"],
    "group": ["Group1"] * 6 + ["Group2"] * 6 + ["Group2", "Group3"],
})
df

In [ ]:
Waterfall(df, x="X", y="Y", category="category", group="group", total=True, subtotals={"C": "Subtotal"}).get_fig()

`total_type="sumtotal"` adds one plain total bar per group instead of a stacked total:

In [ ]:
Waterfall(df, x="X", y="Y", category="category", group="group", total=True, total_type="sumtotal").get_fig()

## 6. Colours

The `color` argument says what drives the colours: the category column (default if there is one), the group column, or `"direction"` (increase / decrease / total; default for charts without category and group). Colours are assigned per value, so a category has the same colour in every group.

`color_discrete_map` sets colours per value; values not in the map get colours from `color_discrete_sequence` (default: the colorway of the plotly template). `pattern_shape` adds fill patterns as a second visual cue, e.g. colour by category and pattern by group:

In [ ]:
Waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    color_discrete_map={"one": "#2a78d6", "two": "#eb6834", "three": "#1baf7a"},
    pattern_shape="group",
).get_fig()

Colour by group instead:

In [ ]:
Waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    color="group",
    color_discrete_map={"Group1": "steelblue", "Group2": "darkorange", "Group3": "seagreen"},
).get_fig()

For full control, keys can also be `(group, category)` tuples. They take precedence over plain values:

In [ ]:
colors = {
    "one": "red", "two": "blue", "three": "green",
    ("Group2", "one"): "salmon", ("Group2", "two"): "lightskyblue",
}
Waterfall(df, x="X", y="Y", category="category", group="group", total=True, color_discrete_map=colors).get_fig()

Direction colours also work with categories and groups. Combine them with a pattern so that the categories stay distinguishable; `color_discrete_map` keys `"increase"`, `"decrease"` and `"total"` override the default green / red / grey:

In [ ]:
Waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    color="direction", pattern_shape="category",
).get_fig()

Keys of `color_discrete_map` that match no value raise a warning, so typos do not go unnoticed.

The older `colors` argument still works but is deprecated and will be removed in version 1.0:

| Old `colors=` | New |
|---|---|
| `{"one": "red", "two": "blue"}` (per category) | `color_discrete_map={"one": "red", "two": "blue"}` |
| `{"Group1": "red", "Group2": "blue"}` (per group) | `color="group", color_discrete_map={"Group1": "red", "Group2": "blue"}` |
| `{"Group1": {"one": "red"}}` (per group and category) | `color_discrete_map={("Group1", "one"): "red"}` |

## 7. Labels, number format and hover

By default every bar is labelled with its change (signed, e.g. `+9`) or, for totals, its value. Numbers get thousands separators and up to two decimals. `number_prefix` / `number_suffix` add units (the sign stays in front: `-$5M`), `number_format` takes a Python format spec such as `",.1f"` or a function. `title` and `labels` work like in plotly express. Hover over a bar to see the step, its category and the running total.

In [ ]:
Waterfall(
    df_cash, x="X", y="Y", total=True,
    number_prefix="$", number_suffix="M",
    title="Cash bridge", labels={"Y": "Cashflow [MUSD]", "X": "Line item"},
).get_fig()

A function as `number_format`, here to show thousands as "k":

In [ ]:
df_k = df_cash.assign(Y=df_cash.Y * 1000)
Waterfall(df_k, x="X", y="Y", total=True, number_format=lambda v: f"{v / 1000:.1f}k", number_prefix="€").get_fig()

`value_labels="segment"` labels every stacked segment instead (segments too small for their label stay unlabelled), `"both"` shows both kinds and `None` hides all labels:

In [ ]:
Waterfall(df, x="X", y="Y", category="category", group="group", total=True, value_labels="segment").get_fig()

## 8. Layout and styling

`barwidth`, `gap` (between the bars of different groups) and `connector_line` (plotly line properties) change the geometry. The result is a regular plotly figure, so everything else — size, legend, fonts, templates — is set with plotly's own methods:

In [ ]:
fig = Waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    barwidth=0.25, gap=0.05,
    connector_line={"color": "black", "width": 1, "dash": "dot"},
).get_fig()
fig.update_layout(template="simple_white", height=450, legend={"orientation": "h", "y": -0.15})